# PanGBank Tutorial: AMR Gene Analysis in *Acinetobacter baumannii*

*Acinetobacter baumannii* is a nosocomial opportunistic pathogen ranked by the WHO as a critical-priority organism for new antibiotic development. The species accumulates antimicrobial resistance (AMR) genes at an exceptional rate through horizontal gene transfer, primarily via genomic islands that integrate at specific chromosomal loci.

This tutorial demonstrates how **PanGBank** and **PPanGGOLiN** can be used to characterise the distribution of AMR genes at the pangenome scale, and how a newly sequenced isolate can be placed into an existing pangenome context without recomputing it from scratch.

**Part 1 — AMR Gene Distribution in the Pangenome**  
A pre-computed *A. baumannii* pangenome is retrieved from PanGBank, its gene families are annotated with AMRFinderPlus, and the partition of AMR genes between the persistent, shell and cloud genome is examined. Insertion hotspots are then analysed to identify the chromosomal loci most frequently associated with antimicrobial resistance

**Part 2 — Projection of a New Genome**  
A canine *A. baumannii* isolate recovered is projected onto the existing pangenome. Each gene is assigned to a pangenome family and their chromosomal loci are matched against the known spot catalogue. RGP gene content is then compared to pangenome RGPs using the Jaccard index to identify the closest relatives in the collection.

> **Before starting**, run the first code cell below once to set up the environment.

In [ ]:
%%capture pangbank_tutorials_init_logs

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
from shutil import which
from pathlib import Path

conda_command = ""
if IN_COLAB:
    !pip install pyvis networkx pygenomeviz pangbank-api[sdk]==0.5.0
    !apt install git-lfs
    !git clone --branch tuto_paper https://github.com/labgem/PanGBank-tutorial.git
    !ln -s PanGBank-tutorial/tutorials/article_use_case/ppanggolin_genome_output ppanggolin_genome_output
    !ln -s PanGBank-tutorial/tutorials/article_use_case/amrfinder_result.tsv amrfinder_result.tsv
    !ln -s PanGBank-tutorial/tutorials/article_use_case/ppanggolin_output ppanggolin_output
    !ln -s PanGBank-tutorial/tutorials/article_use_case/projection_output projection_output
    !curl -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets
    !curl -o dataformat 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/dataformat'
    !chmod +x datasets dataformat

---

## Overview: Precomputed Data

The command-line steps described in each part can be **computationally intensive** and not always suited to a notebook environment.

> **All outputs have been precomputed and are already provided in this repository.** You do not need to run any shell command to follow the tutorial. The step descriptions are included for transparency and reproducibility.

| File / Directory | Produced by | Description |
|---|---|---|
| `amrfinder_result.tsv` | Part 1 – Step 3 | AMRFinder annotations for all pangenome gene families |
| `ppanggolin_output/` | Part 1 – Step 5 | Pangenome flat-file exports (partitions, RGPs, spots, modules) |
| `ppanggolin_genome_output/` | Part 1 – AbaR1 section | Per-genome GFF with pangenome annotations, used for the AbaR1 linear visualisation |
| `projection_output/` | Part 2 – Step 7 | All projection outputs for the canine isolate |

---

## Step 1: Download Pangenome from PanGBank

Search and download the *Acinetobacter baumannii* pangenome from the `GTDB_refseq` collection on [PanGBank](https://pangbank.genoscope.cns.fr/). The `--release-version` flag pins the download to a specific data release to ensure reproducibility.

#### Commands

```bash
pangbank search-pangenomes \
    --collection GTDB_refseq \
    --taxon "s__Acinetobacter baumannii" \
    --download \
    --release-version 2.0.0
```

| | File | Description |
|---|---|---|
| **Input** | — | Queries the PanGBank API (no local file required) |
| **Output** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Pangenome in HDF5 format |
---

Once downloaded, inspect the pangenome content with:

```bash
ppanggolin info \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --content
```

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | The downloaded pangenome HDF5 file |
| **Output** | — | Prints a summary of pangenome statistics to stdout (genome count, gene family count, partition sizes, RGPs, spots, modules) |

## Step 2: Extract Gene Family Sequences

Export one representative protein sequence per gene family from the pangenome. This FASTA file is used as input to AMRFinderPlus in the next step. Running AMRFinder at the family level rather than per genome means each hit maps directly to a pangenome family, avoiding redundant annotation of the same gene across thousands of genomes.

#### Command

```bash
ppanggolin fasta \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --prot_families all \
    --compress \
    -f \
    -o families_faa_output
```

| Flag | Meaning |
|---|---|
| `--prot_families all` | Export sequences for all gene families, regardless of partition |
| `--compress` | gzip the output FASTA file |
| `-f` | Overwrite the output directory if it already exists |
| `-o families_faa_output` | Output directory |

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | The pangenome HDF5 file |
| **Output** | `families_faa_output/all_protein_families.faa.gz` | Gzip-compressed FASTA. One representative protein sequence per gene family, with the family ID as the sequence header |

## Step 3: Annotate AMR Genes with AMRFinderPlus

Run [AMRFinderPlus](https://github.com/ncbi/amr) on the gene family protein sequences to identify antimicrobial resistance (AMR), virulence, and stress response genes. Because the input sequences represent gene families (not individual genomes), each hit maps directly to a pangenome family.

#### Command

```bash
amrfinder \
    -p families_faa_output/all_protein_families.faa.gz \
    --plus \
    --threads 8 \
    -o amrfinder_result.tsv
```

| Flag | Meaning |
|---|---|
| `-p` | Protein FASTA input |
| `--plus` | Also report virulence factors and stress response genes |
| `--threads 8` | Number of parallel threads |
| `-o` | Output file |

| | File | Description |
|---|---|---|
| **Input** | `families_faa_output/all_protein_families.faa.gz` | Representative protein sequences for all gene families |
| **Output** | `amrfinder_result.tsv` | Tab-separated table with one row per annotated gene family. Columns include element symbol, element name, drug class, subclass, detection method, and alignment statistics |

## Step 4: Embed AMRFinder Annotations into the Pangenome

Embed the AMRFinder results directly into the pangenome HDF5 file so that each gene family carries its AMR metadata. Once embedded, downstream tools (e.g. `ppanggolin write_genomes`, `ppanggolin projection`) automatically propagate these annotations to individual genome outputs.

The AMRFinder TSV requires minor header normalisation before import:
- `%` characters are replaced with `Prct` (avoids column-name parsing issues)
- spaces in column names are replaced with underscores
- the `Protein id` column is renamed to `families` to match the key PPanGGOLiN expects

#### Commands

```bash
# Normalise the AMRFinder TSV header in-place
sed -i 's/\%/Prct/g' amrfinder_result.tsv
sed -i '1s/ /_/g' amrfinder_result.tsv
sed -i '1s/\bProtein_id\b/families/' amrfinder_result.tsv

# Embed the annotations into the pangenome
ppanggolin metadata \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --metadata amrfinder_result.tsv \
    --source amrfinder \
    --assign families
```

| Flag | Meaning |
|---|---|
| `--metadata` | Annotation TSV to import, must contain a column matching the `--assign` target |
| `--source amrfinder` | Label attached to this metadata block inside the HDF5 file |
| `--assign families` | Attach metadata rows to gene families, matched by family ID |

| | File | Description |
|---|---|---|
| **Input** | `amrfinder_result.tsv` | AMRFinder annotation table (after header normalisation) |
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | The pangenome HDF5 file |
| **Output** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Same HDF5 file, updated in-place. Gene families now carry `amrfinder` metadata attributes |

## Step 5: Export Pangenome Data

Write flat-file outputs from the pangenome for downstream analysis. Each flag activates a different output type.

#### Command

```bash
ppanggolin write_pangenome \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --spots \
    --regions \
    --families_tsv \
    --partitions \
    --regions_families \
    --modules \
    --spot_modules \
    -o ppanggolin_output \
    -f
```

| Flag | Meaning |
|---|---|
| `--spots` | Export spot assignments for each RGP |
| `--regions` | Export RGP coordinates per genome |
| `--families_tsv` | Export the gene family membership table |
| `--partitions` | Export one text file per partition (persistent, shell, cloud) listing family IDs |
| `--regions_families` | Export the mapping between RGPs and their constituent gene families |
| `--modules` | Export functional module composition |
| `--spot_modules` | Export which modules are found in which spots |
| `-f` | Overwrite the output directory if it already exists |

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | The pangenome HDF5 file (with embedded AMRFinder metadata from Step 4) |
| **Output** | `ppanggolin_output/gene_families.tsv` | Gene family membership — maps each gene (per genome) to its family ID |
| **Output** | `ppanggolin_output/regions_of_genomic_plasticity.tsv` | RGP coordinates and genome assignments |
| **Output** | `ppanggolin_output/rgp_families.tsv` | Gene families belonging to each RGP |
| **Output** | `ppanggolin_output/spots.tsv` | RGP-to-spot assignments |
| **Output** | `ppanggolin_output/partitions/persistent.txt` | Family IDs assigned to the Persistent partition |
| **Output** | `ppanggolin_output/partitions/shell.txt` | Family IDs assigned to the Shell partition |
| **Output** | `ppanggolin_output/partitions/cloud.txt` | Family IDs assigned to the Cloud partition |
| **Output** | `ppanggolin_output/functional_modules.tsv` | Gene family to functional module assignments |
| **Output** | `ppanggolin_output/modules_spots.tsv` | Association between functional modules and spots |
| **Output** | `ppanggolin_output/summarize_spots.tsv` | Per-spot summary statistics (RGP count, genome count, etc.) |

---

# Part 1 — AMR Gene Distribution in the Pangenome

## Biological Context

The resistance phenotype of *Acinetobacter baumannii* is determined by two complementary mechanisms. A set of **intrinsic resistance genes** is encoded in the core genome and present in virtually all strains that confer a baseline level of tolerance to multiple antibiotic classes. In addition, strains continuously acquire **mobile resistance genes** through horizontal gene transfer, carried on genomic islands that integrate at specific chromosomal loci.

PPanGGOLiN partitions gene families into three groups based on their prevalence across the genomes of the pangenome:

| Partition | Presence across genomes | 
|---|---|
| **Persistent** | Near-universal | 
| **Shell** | Intermediate frequency | 
| **Cloud** | Rare (strain-specific or near-specific) | 

Overlaying AMRFinder annotations onto these partitions and onto the [spots of insertion](https://ppanggolin.readthedocs.io/en/latest/user/RGP/rgpAnalyses.html#spot-prediction) enables characterisation of which resistance genes are intrinsic versus mobilisable, which drug classes dominate the accessory resistome, and which insertion loci are most strongly associated with resistance gene acquisition.

In [ ]:
import pandas as pd
from pathlib import Path
import plotly.express as px

In [ ]:
resistance_annotation_file = "amrfinder_result.tsv"
df_amrfinder = pd.read_csv(resistance_annotation_file, sep="\t")

df_amrfinder["amrfinder_annotation"] = True
df_amrfinder

## Filter AMR Results

AMRFinderPlus was run with `--plus`, which extends the standard AMR database to include virulence factors, stress response genes, and other resistance-associated elements. The full output comprises 164 annotated families. Entries are restricted to those where `Type == "AMR"`, excluding the 41 hits classified as `STRESS` or `VIRULENCE`. The resulting set of **123 AMR gene families** is used for all downstream analyses.

In [ ]:
arm_filter = df_amrfinder['Type'] == "AMR" 
df_amr = df_amrfinder.loc[arm_filter][['Protein id', 'Element symbol', 'Element name', 'Scope', 'Type',
       'Subtype', 'Class', 'Subclass', 'Method',
       'HMM description', 'amrfinder_annotation']]

df_amr

## Assign Pangenome Partitions to AMR Gene Families

Each gene family belongs to exactly one partition. The partition lists exported in Step 5 are used to assign a partition label (Persistent, Shell, or Cloud) to each of the 123 AMR families.

This classification is central to distinguishing genes encoded in the core genome and present across virtually all strains and genes carried on mobile elements found only in subsets of strains.

In [ ]:
def parse_partition_file(p_file):
    with open(p_file) as fl:
        return [l.strip() for l in fl]
        
shell_fams = parse_partition_file('ppanggolin_output/partitions/shell.txt')
cloud_fams = parse_partition_file('ppanggolin_output/partitions/cloud.txt')
persistent_fams = parse_partition_file('ppanggolin_output/partitions/persistent.txt')

# 'Protein id' is the family representative sequence ID — the same ID used as the FASTA header
# fed to AMRFinder, so it is the natural key linking AMRFinder hits back to pangenome families.
df_amr.loc[df_amr['Protein id'].isin(persistent_fams), 'partition'] = "Persistent"
df_amr.loc[df_amr['Protein id'].isin(cloud_fams), 'partition'] = "Cloud"
df_amr.loc[df_amr['Protein id'].isin(shell_fams), 'partition'] = "Shell"

df_amr

### AMR Gene Family Distribution by Partition

The two charts explore how the 123 AMR gene families are distributed across pangenome partitions.

The **first chart** shows overall counts per partition. The **second** breaks the same data down by drug class.

In [ ]:
partition_to_color = {'Persistent': '#e59c04', 'Shell': '#00d860', 'Cloud': '#79deff'}
partition_order = ["Persistent", "Shell", "Cloud"]

df_amr_rgp_partition_total = df_amr.groupby(["partition"]).agg({
                                    "Protein id":"count"}).reset_index()

fig = px.bar(df_amr_rgp_partition_total, x='partition', y='Protein id', color="partition", 
             color_discrete_map=partition_to_color,
             category_orders={"partition": partition_order},
             title="AMR Gene Families per Partition",
             text='Protein id',
             labels={"Protein id": "# Gene Families<br>with AMR annotation", "partition": "Pangenome Partition"})

fig.update_traces(textfont_size=12, width=0.6)

fig.update_layout(
    autosize=False,
    width=700,
    height=450,
)
fig.show()

In [ ]:
df_amr_rgp_partition = df_amr.groupby(["partition", "Class"]).agg({
                                    "Protein id":"count"}).reset_index()

fig = px.bar(df_amr_rgp_partition, x='Class', y='Protein id', color="partition", 
             color_discrete_map=partition_to_color,
             category_orders={"partition": partition_order},
             title="AMR Gene Families by Class and Partition",
             labels={"Protein id": "# Gene Families", "Class": "AMR Class", "partition": "Pangenome Partition"})
fig.show()

In [ ]:
df_amr_persistent =df_amr.loc[df_amr['partition'] == "Persistent"][['Protein id', "partition", "Element symbol", "Element name", "Type", "Class"]]
df_amr_persistent

These five persistent AMR gene families are well-characterised intrinsic features of *A. baumannii*:

| Gene | Class | Notes |
|---|---|---|
| `blaADC` | Beta-lactam | Chromosomal AmpC cephalosporinase, intrinsic in virtually all *A. baumannii* |
| `blaOXA` (OXA-51 family) | Beta-lactam | Intrinsic chromosomal carbapenemase, a defining marker of the species |
| `amvA` | Efflux | Multidrug efflux pump contributing to intrinsic tolerance |
| `cxpE` | Phenicol | Chloramphenicol efflux transporter |
| `ant(3'')-IIa` | Aminoglycoside | Aminoglycoside-modifying enzyme |

Their persistent status confirms that they are present in essentially every strain regardless of clinical context. All remaining Shell and Cloud AMR families represent mobilisable resistance.

---

## Building the RGP–Spot–AMR Analysis Table

To analyse how AMR genes are distributed across genomic insertion hotspots, we assemble a unified table from four sources:

1. **RGP gene families** (`rgp_families.tsv`) — which gene families are present in each RGP
2. **Spot assignments** (`spots.tsv`) — which spot each RGP belongs to (RGPs not assigned to any spot are labelled `"No spot"`)
3. **Functional modules** (`functional_modules.tsv`) — co-occurring gene families that form functional units
4. **AMR annotations** — the 123-family AMRFinder table built above

Each row of the merged table represents one gene family in one RGP, enriched with its spot, module, and AMR annotation where applicable.

### Load Spot Assignments

In [ ]:
df_spots = pd.read_csv('ppanggolin_output/spots.tsv', sep='\t')
df_spots

### Gene Families per RGP

Each row links one gene family to the RGP it belongs to in a specific genome. This table drives both the spot-level statistics and the Jaccard comparisons in Part 2.

In [ ]:
# read rgps families
rgp_families_file = Path('ppanggolin_output/rgp_families.tsv')
df_rgp_fams = pd.read_csv(rgp_families_file, sep='\t')

df_rgp_fams

### Load Functional Modules

[Functional modules](https://ppanggolin.readthedocs.io/en/latest/user/Modules/moduleAnalyses.html) are sets of gene families that co-occur across RGPs significantly more often than expected by chance — they typically correspond to operons or mobile elements that are transferred as a unit. Linking AMR families to modules can reveal whether resistance genes travel with a consistent set of accessory genes.

In [ ]:
module_families_file = Path("ppanggolin_output/functional_modules.tsv")
df_module_fams = pd.read_csv(module_families_file, sep="\t")

df_module_fams

### Merge All Data Tables

The four tables are joined on their shared keys, producing one row per (RGP, gene family) pair enriched with spot, module, and AMR columns where applicable. The 123 AMR gene families represent a small fraction of the total gene families in the pangenome; the majority of rows will therefore carry no AMR annotation.

In [ ]:
# Left joins throughout: keep every RGP–family pair even when a family has no spot,
# module, or AMR annotation — unmatched rows get NaN, filled to "No spot" where needed.
df_rgp_info_merged = df_rgp_fams.merge(df_spots, on="rgp_id", how="left")
df_rgp_info_merged['spot_id'] = df_rgp_info_merged['spot_id'].fillna("No spot")
df_rgp_info_merged = df_rgp_info_merged.merge(df_module_fams, on="family_id", how="left")
df_rgp_info_merged = df_rgp_info_merged.merge(df_amr, left_on="family_id", right_on="Protein id", how="left")
df_rgp_info_merged

---

# Spot-Level Analysis

A [Region of Genomic Plasticity (RGP)](https://ppanggolin.readthedocs.io/en/latest/user/RGP/rgpAnalyses.html) is a genome-specific cluster of shell and cloud genes. Most of them arise from Horizontal gene transfer (HGT) and correspond to Genomic Islands (GIs).

[Spots of insertion](https://ppanggolin.readthedocs.io/en/latest/user/RGP/rgpAnalyses.html#spot-prediction) group RGPs from **different genomes** that share the same chromosomal insertion locus, identified by conserved **flanking persistent genes**. Crucially, spot membership is based on locus, not on gene content. Two RGPs at the same spot may carry entirely different mobile elements. The number of RGPs assigned to a spot equals the number of genomes in the pangenome that carry an insertion at that locus.

Some **spots** accumulate hundreds of **RGPs** across many strains, indicating recurrent integration at a highly permissive chromosomal locus, while others appear in only one or two genomes. Summarising AMR gene content per spot identifies which chromosomal loci serve as preferential entry points for resistance genes in *A. baumannii*.

## Summary Statistics per Spot

The following metrics are computed for each spot:
- **n_rgps**: number of genomes with an RGP at this locus
- **n_families / n_modules**: total gene-family and module diversity across all RGPs at this spot
- **n_rgps_with_amr**: number of RGPs at this spot that contain at least one AMR gene family
- **prct_rgp_with_amr**: proportion of RGPs at this spot that carry at least one AMR family (%)

In [ ]:
has_amr = df_rgp_info_merged['amrfinder_annotation'].notna()

# Count unique RGPs, families, and modules per spot across all rows
overall = df_rgp_info_merged.groupby('spot_id').agg(
    n_rgps=('rgp_id', 'nunique'),
    n_families=('family_id', 'nunique'),
    n_modules=('module_id', 'nunique')
)

# Same aggregation but restricted to AMR-annotated rows. Gives the number of RGPs
# at each spot that carry at least one AMR gene family
amr_only = df_rgp_info_merged.loc[has_amr].groupby('spot_id').agg(
    n_rgps_with_amr=('rgp_id', 'nunique'),
    n_families_with_amr=('family_id', 'nunique'),
    n_modules_with_amr=('module_id', 'nunique')
)

# Left join: spots with zero AMR RGPs are absent from amr_only, so fillna(0) is correct
spot_summary = overall.join(amr_only, how='left').fillna(0).astype(int).reset_index()

spot_summary['prct_rgp_with_amr'] = (spot_summary['n_rgps_with_amr'] / spot_summary['n_rgps'] * 100).round(2)
spot_summary

### Spot AMR Enrichment Scatter Plot

Only spots with at least one AMR gene family are shown.

Each point represents one spot:

- **X-axis**: number of RGPs, i.e. how many genomes carry an insertion at this locus
- **Y-axis**: percentage of those RGPs that contain at least one AMR gene family
- **Point size**: gene-family count of the largest RGP at this spot
- **Colour**: number of distinct AMR gene families across all RGPs in the spot

In [ ]:
# Exclude spots with no spot assignment and spots with no AMR content
plot_filter = (spot_summary["spot_id"] != "No spot") & (spot_summary["n_rgps_with_amr"] > 0)

# Compute the size of the largest RGP in each spot:
# 1. Count families in each individual RGP
# 2. Merge with spot assignments
# 3. Take the maximum per spot — this reflects the biggest island observed at each locus,
#    rather than the average (which is diluted by the many small RGPs at the same spot)
rgp_size = df_rgp_fams.groupby('rgp_id')['family_id'].count().reset_index()
rgp_size.columns = ['rgp_id', 'n_families_in_rgp']

rgp_size_with_spot = rgp_size.merge(df_spots, on='rgp_id', how='left')
rgp_size_with_spot['spot_id'] = rgp_size_with_spot['spot_id'].fillna('No spot')

max_fam_per_spot = (
    rgp_size_with_spot.groupby('spot_id')['n_families_in_rgp']
    .max()
    .reset_index()
    .rename(columns={'n_families_in_rgp': 'max_families_in_rgp'})
)

spot_summary = spot_summary.merge(max_fam_per_spot, on='spot_id', how='left')

fig = px.scatter(
    spot_summary.loc[plot_filter],
    x="n_rgps",
    y="prct_rgp_with_amr",
    hover_data=spot_summary.columns,
    color="n_families_with_amr",
    size="max_families_in_rgp",
    title="AMR Gene Family Prevalence Across Genomic Spots",
)
fig.update_coloraxes(colorscale="Viridis")

fig.update_layout(
    autosize=False,
    width=1000,
    height=600,
    xaxis_title="# Genomes",
    yaxis_title="% RGPs with AMR gene families",
    coloraxis_colorbar=dict(title="# AMR families")
)

fig.show()

### Identifying the Key AMR Hotspot

From the scatter plot, **spot_47** stands out: a high proportion of its RGPs carry AMR genes. To understand which resistance island this locus corresponds to, we inspect which genomes carry RGPs there and examine the AMR gene content of those RGPs.

In [ ]:
# Inspect spot_47: which genomes carry RGPs here?
focus_spot = "spot_47"

df_pan_rgps_detail = pd.read_csv("ppanggolin_output/regions_of_genomic_plasticity.tsv", sep="\t")
spot_rgp_ids = df_spots[df_spots["spot_id"] == focus_spot]["rgp_id"]
spot_rgp_detail = (df_pan_rgps_detail[df_pan_rgps_detail["region"].isin(spot_rgp_ids)]
                   [["region", "genome", "genes", "length"]]
                   .sort_values("length", ascending=False)
                   .reset_index(drop=True))

row = spot_summary.loc[spot_summary["spot_id"] == focus_spot].iloc[0]
print(f"{focus_spot}: {int(row['n_rgps_with_amr'])}/{int(row['n_rgps'])} RGPs carry AMR "
      f"({row['prct_rgp_with_amr']:.0f}%),  {int(row['n_families_with_amr'])} distinct AMR families")
print()
display(spot_rgp_detail.head(10))

One of the genomes at spot_47 is **GCF_000069245.1** — the assembly accession for *A. baumannii* strain **AYE** ([NCBI](https://www.ncbi.nlm.nih.gov/assembly/GCF_000069245.1/)), in which [Fournier *et al.* (2006, *PLoS Genetics*)](https://pmc.ncbi.nlm.nih.gov/articles/PMC1326220/) first characterised **AbaR1**. By extracting the AMR gene families in GCF_000069245.1 specific RGP at spot_47, we can verify directly that the gene content matches the published description.

In [ ]:
# Zoom in on the GCF_000069245.1 (strain AYE) RGP specifically
aye_genome = "GCF_000069245.1"
aye_row = spot_rgp_detail[spot_rgp_detail["genome"] == aye_genome]
if not aye_row.empty:
    aye_rgp = aye_row.iloc[0]["region"]
    print(f"\nAMR gene families in {aye_genome} RGP ({aye_rgp}, "
          f"{aye_row.iloc[0]['length']:,} bp, {aye_row.iloc[0]['genes']} genes):")
    aye_amr = (df_rgp_info_merged
               .query(f"rgp_id == '{aye_rgp}'")
               .dropna(subset=["amrfinder_annotation"])
               [["Element symbol", "Element name", "Class", "Subclass"]]
               .drop_duplicates()
               .sort_values(["Class", "Element symbol"])
               .reset_index(drop=True))
    display(aye_amr)

---

## Genome-Level Visualisation: The AbaR1 Resistance Island

GCF_000069245.1 (strain AYE) has an ~86 kb RGP at spot_47 encoding carbapenemases, aminoglycoside-modifying enzymes, and sulfonamide resistance genes, and other markers. This matches the description of **AbaR1** (*Acinetobacter baumannii* Resistance Island 1) in [Fournier *et al.* (2006, *PLoS Genetics*)](https://pmc.ncbi.nlm.nih.gov/articles/PMC1326220/), confirming that spot_47 is the AbaR1 locus.

The visualisation below uses [pyGenomeViz](https://github.com/moshi4/pyGenomeViz) to show the AbaR1 region. AMRFinder metadata was embedded into the pangenome in Step 4, so the GFF exported here already carries AMR annotations per CDS.

```bash
ppanggolin write_genomes \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --genomes GCF_000069245.1 \
    --gff \
    --add_metadata \
    -o ppanggolin_genome_output
```

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Pangenome with embedded AMRFinder metadata |
| **Output** | `ppanggolin_genome_output/gff/GCF_000069245.1.gff` | GFF with partition, family, RGP, spot, and AMRFinder attributes per CDS |

Genes are coloured by partition; AMR-carrying genes have an additional red outline:

| Colour | Partition | Red outline |
|---|---|---|
| Orange | Persistent | Carries an AMR annotation |
| Green | Shell | Carries an AMR annotation |
| Sky blue | Cloud | Carries an AMR annotation |

In [ ]:
# Looking at strain AYE (GCF_000069245.1)
from pygenomeviz import GenomeViz
from pygenomeviz.parser import Gff

def find_spot(spot_name: str, gff):
    # Substring match on the 'spot' GFF attribute, finds the first record at spot_47
    for i in gff.all_records:
        if "spot" in i.attrs and "47" in i.attrs["spot"]:
            return i

def extract_features(features, partition, with_amr):
    if not with_amr:
        return [
            f for f in features
            if f.qualifiers["partition"][0] == partition
        ]
    else:
        return [
            f for f in features
            if f.qualifiers["partition"][0] == partition and "family_amrfinder_Subclass" in f.qualifiers
        ]
    
def show_spot(path, spot, label = ""):
    # First pass: load all contigs to discover which contig hosts the spot_47 record
    gff = Gff("ppanggolin_genome_output/gff/GCF_000069245.1.gff")
    spot_regions = find_spot("47", gff)
    seq_id = spot_regions.seqid
    # Second pass: restrict to that contig — pyGenomeViz renders one contig per track
    gff = Gff("ppanggolin_genome_output/gff/GCF_000069245.1.gff", target_seqid=seq_id)
    features = gff.extract_features(feature_type="CDS", target_range=((spot_regions.start-1000, spot_regions.end+1000)))
    
    AMR_FIELDS = [
        "family_amrfinder_Alignment_length",
        "family_amrfinder_Class",
        "family_amrfinder_Closest_reference_accession",
        "family_amrfinder_Closest_reference_name",
        "family_amrfinder_Element_name",
        "family_amrfinder_Element_symbol",
        "family_amrfinder_HMM_accession",
        "family_amrfinder_HMM_description",
        "family_amrfinder_Method",
        "family_amrfinder_Prct_Coverage_of_reference",
        "family_amrfinder_Prct_Identity_to_reference",
        "family_amrfinder_Reference_sequence_length",
        "family_amrfinder_Scope",
        "family_amrfinder_Subclass",
        "family_amrfinder_Subtype",
        "family_amrfinder_Target_length",
        "family_amrfinder_Type",
        "family_amrfinder_metadata_id"
    ]

    gv = GenomeViz()
    
    gv.set_scale_bar(ymargin=0.5)
    target_ranges = ((spot_regions.start-2500, spot_regions.end+500))
    track = gv.add_feature_track(name=gff.name, segments=target_ranges)

    # Collect feature attributes into a flat DataFrame — returned alongside the figure
    # so the caller can display the AMR annotation table (cell below)
    data = []
    for f in features:
        l = []
        qual = f.qualifiers
        l.append(f.id)
        l.append(f.location.start)
        l.append(f.location.end)
        l.append(f.location.strand)
        l.extend([qual["product"][0], qual["partition"][0], qual["family"][0], qual["rgp"][0]])
        l.extend([qual[e][0] if e in qual else None for e in AMR_FIELDS])
        data.append(l)


    df = pd.DataFrame(data, columns=["ID", "start", "end", "strand", "product", "partition", "family", "rgp"] + AMR_FIELDS)
    df_amr = df[df[AMR_FIELDS].notna().any(axis=1)]
    df_no_amr = df[df[AMR_FIELDS].isna().all(axis=1)]

    for segment in track.segments:
    
        cds_features = gff.extract_features(feature_type="CDS", target_range=segment.range)
        
        p_features = extract_features(cds_features, "persistent", False)
        s_features = extract_features(cds_features, "shell", False)
        c_features = extract_features(cds_features, "cloud", False)
        
        p_features_amr = extract_features(cds_features, "persistent", True)
        s_features_amr = extract_features(cds_features, "shell", True)
        c_features_amr = extract_features(cds_features, "cloud", True)
        
        segment.add_features(p_features, fc="orange", lw=0.0)
        segment.add_features(s_features, fc="green", lw=0.0)
        segment.add_features(c_features, fc="skyblue", lw=0.0)
        
        # Draw AMR-carrying genes a second time with a red outline — same coordinates,
        # layered on top so the border is visible without moving the feature
        segment.add_features(p_features_amr, lw=1.0, ec="red", fc="orange")
        segment.add_features(s_features_amr, lw=1.0, ec="red", fc="green", label_type=label)
        segment.add_features(c_features_amr, lw=1.0, ec="red", fc="skyblue", label_type=label)
    
    return gv.plotfig(), df_amr, df_no_amr

In [ ]:
fig, df_amr, df_no_amr = show_spot("ppanggolin_genome_output/gff/GCF_000069245.1.gff", "47")

The table below lists every AMR-annotated gene in the AbaR1 region of strain AYE (the genes outlined in red in the visualisation above) with their drug class and AMRFinder detection details.

In [ ]:
df_amr

---

# Part 2 — Projection of a New Genome onto the Pangenome

## Biological Context

The **projection** feature of PPanGGOLiN enables the annotation of a new, external genome using an existing pangenome **without recomputing it**. Each gene in the incoming genome is assigned to the most similar pangenome gene family (or flagged as novel if no match is found), its Regions of Genomic Plasticity are detected, and those RGPs are matched to known spots based on their flanking persistent genes.

Projection is particularly suited to clinical and surveillance contexts: a single command yields a complete characterisation of a newly sequenced isolate with its partition profile, completeness relative to the pangenome core, RGPs, etc.

Here, an *A. baumannii* canine isolate ([GCF_033191195.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_033191195.1/)) is projected onto the *A. baumannii* pangenome constructed in Part 1. The central question is whether this animal isolate carries the AbaR1 resistance island, a locus so far characterised predominantly in nosocomial human strains.

## Step 6: Download the New Genome

```bash
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/033/191/195/GCF_033191195.1_ASM3319119v1/GCF_033191195.1_ASM3319119v1_genomic.gbff.gz
```

| | File | Description |
|---|---|---|
| **Input** | — | NCBI FTP (no local file required) |
| **Output** | `GCF_033191195.1_ASM3319119v1_genomic.gbff.gz` | Annotated genome in GenBank flat-file format (gzip-compressed) |

## Step 7: Run the Projection

```bash
ppanggolin projection \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --anno GCF_033191195.1_ASM3319119v1_genomic.gbff.gz \
    --gff \
    --proksee \
    -o projection_output
```

| Flag | Meaning |
|---|---|
| `--anno` | Input genome annotation file (GenBank or GFF format) |
| `--gff` | Export an annotated GFF with partition, RGP, spot, and metadata attributes per gene |
| `--proksee` | Export a Proksee-compatible JSON for circular genome visualisation |
| `-o` | Output directory |

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | The existing pangenome (from Part 1) |
| **Input** | `GCF_033191195.1_ASM3319119v1_genomic.gbff.gz` | The new genome to project |
| **Output** | `projection_output/summary_projection.tsv` | Overall statistics: partition counts, RGP/spot/module counts, completeness |
| **Output** | `projection_output/input_genome/gene_to_gene_family.tsv` | Mapping from each gene in the new genome to its pangenome family |
| **Output** | `projection_output/input_genome/sequences_partition_projection.tsv` | Partition assignment per gene |
| **Output** | `projection_output/input_genome/regions_of_genomic_plasticity.tsv` | RGPs detected in the new genome with coordinates |
| **Output** | `projection_output/input_genome/input_genome_rgp_to_spot.tsv` | RGP-to-spot assignments (matched by flanking persistent genes) |
| **Output** | `projection_output/input_genome/modules_in_input_genome.tsv` | Functional modules detected in the new genome with completion level |
| **Output** | `projection_output/input_genome/input_genome_proksee.json` | Proksee JSON for circular visualisation |

> **All outputs have been precomputed and are available in `projection_output/`.**

---

## Projection Results

The projection summary is first examined to assess how well the canine isolate fits within the *A. baumannii* pangenome.

In [ ]:
import pandas as pd
import yaml

with open("projection_output/input_genome/projection_summary.yaml") as f:
    summary = yaml.safe_load(f)

# The YAML first key is literally "Projection_summary:Genome_name" (colon in the key name)
genome_name = summary["Projection_summary:Genome_name"]
print(f"Genome: GCF_033191195.1 (dog isolate)")
print(f"Completeness:  {summary['Completeness']}%")
print(f"Genes:         {summary['Genes']}")
print(f"Persistent:    {summary['Persistent']['genes']} genes / {summary['Persistent']['families']} families")
print(f"Shell:         {summary['Shell']['genes']} genes / {summary['Shell']['families']} families")
print(f"Cloud:         {summary['Cloud']['genes']} genes / {summary['Cloud']['families']} families")
print(f"  Specific:    {summary['Cloud']['specific families']} families (not in any pangenome family)")
print(f"RGPs detected: {summary['RGPs']}")
print(f"Spots matched: {summary['Spots']}")
print(f"Modules found: {summary['Modules']}")
print(f"New spots:     {summary['New_spots']}")

The canine isolate is integrated into the *A. baumannii* pangenome with **99.68 % completeness** (only 2 genome-specific families absent from any pangenome family). All 39 detected RGPs are matched to existing spots in the pangenome, indicating that none of the RGPs in this isolate occupy a chromosomal locus not previously observed in the species.

Inspection of the RGP-to-spot assignment table determines whether **spot_47**, the AbaR1 resistance island locus, is occupied in the canine isolate.

In [ ]:
df_proj_spots = pd.read_csv("projection_output/input_genome/input_genome_rgp_to_spot.tsv", sep="\t")
df_proj_rgps  = pd.read_csv("projection_output/input_genome/regions_of_genomic_plasticity.tsv", sep="\t")

df_proj = df_proj_rgps.merge(df_proj_spots, left_on="region", right_on="region", how="left")
df_proj["spot_id"] = df_proj["spot_id"].fillna("No spot")

# Highlight the AbaR1 locus
print("RGP at spot_47 (AbaR1 locus):")
display(df_proj[df_proj["spot_id"] == "spot_47"][["region", "contig", "start", "stop", "length", "genes", "spot_id"]])

print(f"\nAll {len(df_proj)} RGPs detected in the dog isolate:")
df_proj[["region", "length", "genes", "spot_id"]]

The canine isolate carries **CP136181.1_RGP_12** at **spot_47**. At approximately 22 kb with 18 gene families, the RGP is considerably more compact than the ~86 kb island of strain AYE. This is consistent with the definition of a spot: insertion locus is determined by flanking persistent genes, not by RGP, so structurally distinct mobile elements can occupy the same chromosomal position in different strains.

The presence of an insertion at spot_47 in a canine isolate indicates that this integration locus is not exclusive to nosocomial human strains. The AbaR1 locus appears to be broadly accessible across *A. baumannii* lineages from distinct ecological sources.

## Circular Genome Visualisation

The `--proksee` flag produces a JSON file compatible with [Proksee](https://proksee.ca/) and the [CGView.js](https://js.cgview.ca/) library. The circular map below renders the complete chromosome of the canine isolate with genes coloured by pangenome partition. A dedicated outer track in red highlights genes carrying an AMR annotation from AMRFinder.

| Colour | Meaning |
|---|---|
| Orange | Persistent gene (core genome) |
| Green | Shell gene (intermediate frequency) |
| Blue | Cloud gene (rare / strain-specific) |
| **Red outer ring** | AMR-annotated gene (hover to display element symbol, drug class, and identity) |

The AMR track provides an immediate view of which chromosomal regions concentrate resistance genes. The inner RGP arcs delimit the extent of each detected genomic island.

In [ ]:
import json
from pathlib import Path
from IPython.display import display, HTML

matches = list(Path("projection_output").rglob("input_genome_proksee.json"))

with open(matches[0]) as ff:
    data = json.load(ff)

# Add a legend entry for the AMR track
data['cgview']['legend']['items'].append({
    'decoration': 'arrow',
    'name': 'AMR',
    'swatchColor': '#e74c3c'
})

# Add a dedicated AMR track as a second outside ring (thinner than the Gene ring)
data['cgview']['tracks'].append({
    'dataKeys': 'AMR',
    'dataMethod': 'source',
    'dataType': 'feature',
    'name': 'AMR',
    'position': 'outside',
    'separateFeaturesBy': 'strand',
    'thicknessRatio': 0.5
})

# Duplicate each AMR-annotated gene as a new feature assigned to the AMR track
for feature in list(data['cgview']['features']):
    meta = feature.get('meta', {})
    if any(k.startswith('family_amrfinder_') for k in meta):
        symbol = meta.get('family_amrfinder_Element_symbol', ['AMR'])
        name = symbol[0] if isinstance(symbol, list) else symbol
        data['cgview']['features'].append({
            'contig':  feature['contig'],
            'legend':  'AMR',
            'meta':    meta,
            'name':    name,
            'source':  'AMR',
            'start':   feature['start'],
            'stop':    feature['stop'],
            'strand':  feature.get('strand', 1),
            'type':    'AMR'
        })

json_str = json.dumps(data)
container_id = "cgview-container"

html = f"""
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/cgview/dist/cgview.css">

<div id="{container_id}" style="width:800px; height:600px; border:1px solid #ccc;">
    Loading CGView...
</div>

<script>
(function() {{
    function loadScript(src) {{
        return new Promise(function(resolve, reject) {{
            var s = document.createElement('script');
            s.src = src;
            s.onload = resolve;
            s.onerror = reject;
            document.head.appendChild(s);
        }});
    }}

    Promise.resolve()
        .then(function() {{ return loadScript('https://cdn.jsdelivr.net/npm/d3@7'); }})
        .then(function() {{ return loadScript('https://cdn.jsdelivr.net/npm/cgview/dist/cgview.min.js'); }})
        .then(function() {{
            var jsonData = {json_str};
            var container = document.getElementById('{container_id}');
            container.innerHTML = '';
            var viewer = new CGView.Viewer('{container_id}', {{ width: 800, height: 600 }});
            viewer.io.loadJSON(jsonData);
        }})
        .catch(function(err) {{
            document.getElementById('{container_id}').innerText = 'Failed to load CGView: ' + err;
        }});
}})();
</script>
"""

display(HTML(html))

---

## Comparing the Dog's Genomic Islands to the Pangenome

Spot membership is determined solely by flanking genomic context. Two RGPs assigned to the same spot may carry entirely different gene content. To measure **content similarity** independently of locus identity, the **Jaccard index** is computed between the gene-family set of each canine RGP and every RGP in the pangenome:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

A score of 1 indicates complete gene-family identity; a score of 0 indicates no shared families. This comparison identifies which pangenome strains carry the most similar versions of the canine isolate's RGP, and whether those nearest neighbours occupy the same insertion locus.

In [ ]:
from pygenomeviz.parser import Gff

# GffRecord.attrs values are list[str] (comma-split per GFF3 spec),
# and .type is the feature type string (CDS, gene, …)
gff = Gff("projection_output/input_genome/input_genome.gff")

dog_rgp_to_families = {}
dog_rgp_has_amr = set()

for record in gff.all_records:
    if record.type != "CDS":
        continue
    if "rgp" not in record.attrs or "family" not in record.attrs:
        continue
    rgp = record.attrs["rgp"][0]
    dog_rgp_to_families.setdefault(rgp, set()).add(record.attrs["family"][0])
    # family_amrfinder_Type is propagated from the pangenome metadata by ppanggolin projection
    if record.attrs.get("family_amrfinder_Type", [None])[0] == "AMR":
        dog_rgp_has_amr.add(rgp)

print(f"Dog RGPs with gene-family data: {len(dog_rgp_to_families)}")
print(f"Dog RGPs with an AMR-annotated gene: {len(dog_rgp_has_amr)}")
if dog_rgp_has_amr:
    for rgp in sorted(dog_rgp_has_amr):
        print(f"  {rgp}: {len(dog_rgp_to_families[rgp])} families")

# If no AMR RGP is found (all AMR genes are persistent and not assigned to any RGP),
# fall back to the RGP at spot_47
dog_amr_rgps = {rgp: dog_rgp_to_families[rgp] for rgp in dog_rgp_has_amr}
if not dog_amr_rgps:
    print("\nNo shell/cloud AMR genes found in dog RGPs.")
    print("Falling back to CP136181.1_RGP_12 (spot_47, AbaR1 locus) for Jaccard comparison.")
    dog_amr_rgps = {"CP136181.1_RGP_12": dog_rgp_to_families["CP136181.1_RGP_12"]}

In [ ]:
# Build pangenome RGP -> set of families (already loaded as df_rgp_fams)
pan_rgp_families = df_rgp_fams.groupby("rgp_id")["family_id"].apply(set)

# Load pangenome RGP -> genome mapping
df_pan_rgps = pd.read_csv("ppanggolin_output/regions_of_genomic_plasticity.tsv", sep="	")
rgp_to_genome = df_pan_rgps.set_index("region")["genome"].to_dict()

def jaccard(a: set, b: set) -> float:
    i = len(a & b)
    return 0.0 if i == 0 else i / len(a | b)

results = []
for dog_rgp, dog_fams in dog_amr_rgps.items():
    for pan_rgp, pan_fams in pan_rgp_families.items():
        j = jaccard(dog_fams, pan_fams)
        if j > 0:
            results.append({"dog_rgp": dog_rgp, "pan_rgp": pan_rgp, "jaccard": j,
                            "genome": rgp_to_genome.get(pan_rgp, ""),
                            "dog_families": len(dog_fams), "pan_families": len(pan_fams),
                            "shared_families": len(dog_fams & pan_fams)})

df_jaccard = (pd.DataFrame(results)
              .sort_values("jaccard", ascending=False)
              .reset_index(drop=True))

# Annotate with spot assignment
df_jaccard = df_jaccard.merge(df_spots.rename(columns={"rgp_id": "pan_rgp"}),
                              on="pan_rgp", how="left")
df_jaccard["spot_id"] = df_jaccard["spot_id"].fillna("No spot")

print(f"Pangenome RGPs with non-zero Jaccard: {len(df_jaccard)}")
df_jaccard[["dog_rgp", "genome", "pan_rgp", "jaccard", "shared_families", "pan_families", "spot_id"]].head(20)

## Linking Results to PanGBank

Genome accessions from the Jaccard table are resolved to PanGBank genome URLs using the [PanGBank SDK](https://github.com/labgem/PanGBank-api/tree/main/pangbank_api/sdk). Two identifiers are required:

- **Pangenome ID**: retrieved by querying PanGBank for the *A. baumannii* GTDB-Refseq pangenome
- **Genome ID**: retrieved per accession from the pangenome's genome list

The resulting URL follows the form:
```
https://pangbank.genoscope.cns.fr/pangenome/{pangenome_id}/genome/{genome_id}
```

In [ ]:
from pangbank_api.sdk import PanGBankClient

PANGBANK_WEB = "https://pangbank.genoscope.cns.fr"
top_n = 10

with PanGBankClient() as client:
    # Resolve the pangenome ID for this A. baumannii pangenome
    pangenomes = client.pangenomes.list(
        taxon_name="s__Acinetobacter baumannii",
        collection_name="GTDB_refseq",
        only_latest_release=True,
        limit=1,
    )
    pangenome_id = pangenomes[0].id
    print(f"Pangenome ID: {pangenome_id}")

    # Collect unique genome names from the top-N results across all dog RGPs
    unique_genomes = set()
    for _, grp in df_jaccard.groupby("dog_rgp"):
        unique_genomes.update(grp["genome"].head(top_n).tolist())

    # Batch-resolve genome_id for each unique genome
    genome_id_map = {}
    for gname in sorted(unique_genomes):
        links = client.pangenomes.list_genomes(pangenome_id, genome_name=gname, limit=1)
        if links:
            genome_id_map[gname] = links[0].genome_id

print(f"Resolved {len(genome_id_map)}/{len(unique_genomes)} genome IDs")

### Top-10 Similar Pangenome RGPs 

Four RGPs carrying AMR-annotated genes are identified in the canine isolate (at spots 11, 7, 47, and 31). For each, the table below reports the 10 most similar pangenome RGPs ranked by Jaccard index, together with the genome of origin and spot assignment.

**Genome names are clickable links** that open the corresponding genome page on **PanGBank**.

In [ ]:
def genome_link(gname):
    gid = genome_id_map.get(gname)
    if gid:
        return f'<a href="{PANGBANK_WEB}/pangenome/{pangenome_id}/genome/{gid}" target="_blank">{gname}</a>'
    return gname

for dog_rgp, grp in df_jaccard.groupby("dog_rgp"):
    # Look up which spot the dog RGP maps to for the section header
    spot_assigned = (df_proj_spots.set_index("region").loc[dog_rgp, "spot_id"]
                     if dog_rgp in df_proj_spots["region"].values else "No spot")

    top = grp.head(top_n).copy().reset_index(drop=True)
    top["genome"] = top["genome"].apply(genome_link)

    cols = ["genome", "pan_rgp", "jaccard", "shared_families", "pan_families", "spot_id"]
    html_table = top[cols].to_html(escape=False, index=False)

    print(f"{dog_rgp}  ->  spot: {spot_assigned}  |  {len(grp)} pangenome RGPs with shared families")
    display(HTML(html_table))
    print()

### Contextualising the AbaR1 Perfect-Match Strains

The 6 strains sharing a gene-for-gene identical AbaR1 island with the canine isolate are queried using the [NCBI Datasets](https://www.ncbi.nlm.nih.gov/datasets/docs/v2/download-and-install/) tool to retrieve biosample metadata (host organism, isolation source, and geographic origin).

In [ ]:
import subprocess, json

# Retrieve accessions of the perfect-match AbaR1 strains (Jaccard = 1.0 for RGP_12)
abar1_rgp = "CP136181.1_RGP_12"
exact_matches = (df_jaccard
                 .query(f"dog_rgp == '{abar1_rgp}' and jaccard == 1.0")["genome"]
                 .tolist())

# Query NCBI datasets CLI for biosample metadata
result = subprocess.run(
    ["datasets", "summary", "genome", "accession", ",".join(exact_matches)],
    capture_output=True, text=True, check=True,
)
ncbi_data = json.loads(result.stdout)

# Extract strain, host, isolation source, and geographic origin
rows = []
for r in ncbi_data.get("reports", []):
    attrs = {a["name"]: a.get("value", "")
             for a in r.get("assembly_info", {}).get("biosample", {}).get("attributes", [])}
    rows.append({
        "genome":           r["accession"],
        "strain":           r.get("organism", {}).get("infraspecific_names", {}).get("strain", ""),
        "host":             attrs.get("host", ""),
        "isolation_source": attrs.get("isolation_source", ""),
        "country":          attrs.get("geo_loc_name", ""),
    })

df_abar1_meta = (pd.DataFrame(rows)
                 .merge(df_jaccard[df_jaccard["dog_rgp"] == abar1_rgp][["genome", "pan_rgp"]],
                        on="genome"))

display(df_abar1_meta[["genome", "strain", "host", "isolation_source", "country", "pan_rgp"]])

The Jaccard comparison reveals distinct similarity profiles across the four AMR RGPs of the canine isolate.

**RGP_12 — spot_47 (AbaR1 locus)**  
A Jaccard score of 1.0 is observed with 6 pangenome RGPs: the AbaR1 island of the canine isolate shares all 17 gene families with these strains. As shown in the table above, all 6 matching strains are **human clinical isolates** recovered from respiratory infections (sputum, phlegm, or broncho-alveolar lavage), originating from geographically distant locations: China (Hebei, Hunan, Anhui), the United States, and Vietnam. 

**RGP_0 — spot_11 and RGP_1 — spot_7**  
Both RGPs are substantially larger (142 and 123 gene families, respectively) and display near-perfect Jaccard scores (~0.99), corresponding to a single gene-family difference from their closest pangenome neighbours. The large number of pangenome RGPs with partial content overlap (>10,000 each) reflects the wide distribution of the gene families these RGPs carry. Top hits consistently map to the same spot as the canine RGP, indicating well-established, conserved integration loci.

**RGP_5 — spot_31**  
A slightly lower Jaccard score (0.947 for the top hit; 18 families shared out of 19) points to a minor but consistent divergence in gene content relative to the closest pangenome representatives.

Taken together, these results indicate that the resistance repertoire of the canine isolate is not unique. Each AMR RGP is shared, at the same chromosomal locus, with identifiable strains elsewhere in the pangenome. The identity between the canine AbaR1 variant and human clinical isolates from multiple countries highlights the epidemiological relevance of the PanGBank projection workflow for tracing resistance island dissemination across host boundaries. The PanGBank links in each table provide direct access to those genomes for further comparative analysis.